# 12 — Raw Activation Extraction (Objects + Checkers)

**Purpose:** Extract raw activation statistics (`activation_sum`, `firing_count`) for both object prompts and checker prompts. These raw stats can be re-thresholded at any epsilon/consistency combination without re-running the model.

In [ ]:
# Cell 1 – Dependency setup (circuit_sparsity injection + pip installs)
import subprocess, sys, os, types, importlib.util, glob, inspect

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "huggingface_hub"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Locate real gpt.py & hook_utils.py ───────────────────────────────────
tm_base     = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/openai")
tm_gpt_hits = glob.glob(os.path.join(tm_base, "*/gpt.py"))

if tm_gpt_hits:
    gpt_path  = tm_gpt_hits[0]
    hook_path = os.path.join(os.path.dirname(gpt_path), "hook_utils.py")
    print(f"Using transformers_modules cache: {os.path.dirname(gpt_path)}")
else:
    from huggingface_hub import hf_hub_download
    gpt_path  = hf_hub_download("openai/circuit-sparsity", "gpt.py")
    hook_path = hf_hub_download("openai/circuit-sparsity", "hook_utils.py")
    print("Using hf_hub_download (first run)")

hf_dir = os.path.dirname(gpt_path)

cs_pkg = types.ModuleType("circuit_sparsity")
cs_pkg.__path__ = [hf_dir]
cs_pkg.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity"] = cs_pkg

hook_spec = importlib.util.spec_from_file_location("circuit_sparsity.hook_utils", hook_path)
hook_mod  = importlib.util.module_from_spec(hook_spec)
hook_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.hook_utils"] = hook_mod
hook_spec.loader.exec_module(hook_mod)
cs_pkg.hook_utils = hook_mod

gpt_spec = importlib.util.spec_from_file_location("circuit_sparsity.gpt", gpt_path)
gpt_mod  = importlib.util.module_from_spec(gpt_spec)
gpt_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.gpt"] = gpt_mod
gpt_spec.loader.exec_module(gpt_mod)

_RealGPTConfig = gpt_mod.GPTConfig
_valid_params  = set(inspect.signature(_RealGPTConfig.__init__).parameters) - {"self"}

class _PatchedGPTConfig(_RealGPTConfig):
    def __init__(self, **kwargs):
        super().__init__(**{k: v for k, v in kwargs.items() if k in _valid_params})

gpt_mod.GPTConfig = _PatchedGPTConfig
cs_pkg.gpt = gpt_mod

print(f"circuit_sparsity ready — GPTConfig patched")

In [ ]:
# Cell 2 – Configuration
MODEL_ID = "openai/circuit-sparsity"
OBJECT_PROMPTS = "11A_object_prompts.parquet"
CHECKER_PROMPTS = "11B_checker_prompts.parquet"
OBJECT_OUTPUT = "12_object_activations.h5"
CHECKER_OUTPUT = "12_checker_activations.h5"
N_LAYERS = 8
CHECKPOINT_EVERY = 200

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")
print(f"Object prompts: {OBJECT_PROMPTS}")
print(f"Checker prompts: {CHECKER_PROMPTS}")

In [ ]:
# Cell 3 – Imports
import numpy as np, pandas as pd, torch, h5py
from tqdm.auto import tqdm
import logging, re

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("notebook12")
print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Cell 4 – Load model & tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=torch.float16,
).to(DEVICE).eval()

print(f"Model loaded on {DEVICE} | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 5 – Register hooks
from module2.extraction import ActivationExtractor

mlp_pattern = None
seen_lids = set()
for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        lid = int(m.group(2))
        seen_lids.add(lid)
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)

print(f"Pattern: {mlp_pattern} | Layers: {sorted(seen_lids)}")

extractor = ActivationExtractor(
    model=model, tokenizer=tokenizer, device=DEVICE,
    n_layers=N_LAYERS, use_hook_recorder=False,
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

# Sanity check
acts = extractor.extract('x = 42')
for lid, vec in sorted(acts.items()):
    print(f"  Layer {lid}: shape={tuple(vec.shape)}, mean |act|={vec.abs().mean().item():.6f}")

In [ ]:
# Cell 6 – Extract OBJECT activations
from module2.binarization import RawActivationCollector
from module2.io_utils import save_activations_hdf5
import pickle

df_obj = pd.read_parquet(os.path.join(DATA_DIR, OBJECT_PROMPTS))
print(f"Object prompts: {len(df_obj)} rows, {df_obj.groupby(['ast_node','builtin_obj']).ngroups} pairs")

collector = RawActivationCollector(n_layers=N_LAYERS)
grouped = df_obj.groupby(["ast_node", "builtin_obj"])
all_pairs = list(grouped.groups.keys())

pair_activations = {}
for idx, (ast_n, blt_o) in enumerate(tqdm(all_pairs, desc="Object extraction")):
    prompts = df_obj.loc[grouped.groups[(ast_n, blt_o)], "prompt_text"].tolist()
    raw = collector.collect(extractor, prompts)
    pair_activations[(ast_n, blt_o)] = raw

# Save
obj_output_path = os.path.join(DATA_DIR, OBJECT_OUTPUT)
metadata = {
    "model_id": MODEL_ID,
    "n_layers": N_LAYERS,
    "n_pairs": len(pair_activations),
    "input_file": OBJECT_PROMPTS,
    "ast_nodes": sorted(set(a for a, _ in pair_activations)),
    "builtin_objs": sorted(set(b for _, b in pair_activations)),
}
save_activations_hdf5(obj_output_path, pair_activations, metadata)
print(f"Saved: {obj_output_path}")

In [ ]:
# Cell 7 – Extract CHECKER activations
df_chk = pd.read_parquet(os.path.join(DATA_DIR, CHECKER_PROMPTS))
print(f"Checker prompts: {len(df_chk)} rows, {df_chk['object'].nunique()} objects")

checker_activations = {}
for obj_key in tqdm(sorted(df_chk["object"].unique()), desc="Checker extraction"):
    prompts = df_chk[df_chk["object"] == obj_key]["prompt_text"].tolist()
    if len(prompts) == 0:
        continue
    raw = collector.collect(extractor, prompts)
    # Split obj_key on __ to create tuple key for save_activations_hdf5 compatibility
    parts = obj_key.split("__", 1)
    checker_activations[(parts[0], parts[1])] = raw

# Save
chk_output_path = os.path.join(DATA_DIR, CHECKER_OUTPUT)
chk_metadata = {
    "model_id": MODEL_ID,
    "n_layers": N_LAYERS,
    "n_pairs": len(checker_activations),
    "input_file": CHECKER_PROMPTS,
}
save_activations_hdf5(chk_output_path, checker_activations, chk_metadata)
print(f"Saved: {chk_output_path}")

In [ ]:
# Cell 8 – Verification
for label, path in [("Object", obj_output_path), ("Checker", chk_output_path)]:
    with h5py.File(path, "r") as f:
        counts = [0]
        def count(name, obj):
            if isinstance(obj, h5py.Dataset):
                counts[0] += 1
        f.visititems(count)
        print(f"{label}: {path}")
        print(f"  Datasets: {counts[0]}")

In [ ]:
# Cell 9 – Cleanup
extractor.remove_hooks()
print("Hooks removed.")
print(f"\n12 complete.")
print(f"  Object activations: {obj_output_path}")
print(f"  Checker activations: {chk_output_path}")